In [1]:
# from xopr_gline import xopr_utils
import numpy as np
import xarray as xr
import hvplot.xarray
import matplotlib.pyplot as plt
import scipy.constants
from scipy import signal
import pandas as pd
import geopandas as gpd
import xopr.opr_access
import xopr.geometry
import dask
from dask.distributed import LocalCluster
import cartopy.crs as ccrs
import geoviews.feature as gf
import time
import requests
from scipy.optimize import curve_fit

from xopr_gline.xopr_utils import extract_layer_peak_power, surface_bed_reflection_power, get_basal_layer_wgs84
from xopr_gline.empirical import erf_topography_model, get_derivatives

In [2]:
cresis_fl_path = '../data/cresis_useable_flight_lines.geojson'
cresis_fl = gpd.read_file(cresis_fl_path)
cresis_fl

,fid,OBJECTID,Name,FolderPath,SymbolID,AltMode,Base,Clamped,Extruded,Snippet,PopupInfo,Shape_Leng,geometry
0,1,1,20160509_07_010,20160509_07_010.kmz,0,-1,0.0,0,0,None,None,2.977177,"MULTILINESTRING Z ((-24.39078 79.0248 0, -21.9..."
1,2,2,20160509_10_001,20160509_10_001.kmz,0,-1,0.0,0,0,None,None,2.595372,"MULTILINESTRING Z ((-19.43633 79.55342 0, -20...."
2,3,3,20160519_03_016,20160519_03_016.kmz,0,-1,0.0,0,0,None,None,1.331803,"MULTILINESTRING Z ((-30.40571 72.71069 0, -30...."
3,4,4,20160519_03_026,20160519_03_026.kmz,0,-1,0.0,0,0,None,None,1.019261,"MULTILINESTRING Z ((-30.83517 73.38396 0, -30...."
4,5,5,19950524_01_011,19950524_01_011.kmz,0,-1,0.0,0,0,None,None,2.440836,"MULTILINESTRING Z ((-47.7925 69.45962 0, -47.8..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
331,332,76,20140512_01_027,20140512_01_027.kmz,0,-1,0.0,0,0,None,None,1.048882,"MULTILINESTRING Z ((-66.90451 77.4937 0, -66.7..."
332,333,77,20140512_01_028,20140512_01_028.kmz,0,-1,0.0,0,0,None,None,1.175858,"MULTILINESTRING Z ((-66.62773 77.74019 0, -66...."
333,334,26,20140424_01_022,20140424_01_022.kmz,0,-1,0.0,0,0,None,None,0.779269,"MULTILINESTRING Z ((-32.60986 67.60396 0, -32...."
334,335,31,20140424_01_043,20140424_01_043.kmz,0,-1,0.0,0,0,None,None,1.090071,"MULTILINESTRING Z ((-38.18128 66.36619 0, -38...."


In [3]:
opr = xopr.opr_access.OPRConnection(cache_dir="/tmp")

In [10]:
cresis_fl.iloc[0]["Name"]

'20160509_07_010'

In [11]:
# Select a segment

selected_segment = cresis_fl.iloc[0]["Name"][:-4]
print(f"Selected segment: {selected_segment}")

# Query frames
stac_items = opr.query_frames(
segment_paths=[selected_segment]
)
print(f"Found {len(stac_items)} frames")
stac_items = stac_items.iloc[:]
stac_items

Selected segment: 20160509_07
Found 10 frames


,collection,geometry,properties,assets,bbox,id,links,stac_extensions,stac_version,type
stac_item_id,,,,,,,,,,
Data_20160509_07_001,2016_Greenland_P3,"LINESTRING (-43.50827 79.71523, -43.336 79.716...",{'datetime': '2016-05-09T12:35:04.367510+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-43.508269243647845, 79.71522892639231, -41.0...",Data_20160509_07_001,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_002,2016_Greenland_P3,"LINESTRING (-41.00785 79.73782, -38.49748 79.7...",{'datetime': '2016-05-09T12:41:02.841628+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-41.007853135364115, 79.73782265706829, -38.4...",Data_20160509_07_002,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_003,2016_Greenland_P3,"LINESTRING (-38.4957 79.74134, -35.98824 79.72...",{'datetime': '2016-05-09T12:46:58.471549+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-38.49569842835945, 79.72515432589267, -35.98...",Data_20160509_07_003,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_004,2016_Greenland_P3,"LINESTRING (-35.98648 79.72513, -33.4915 79.69...",{'datetime': '2016-05-09T12:52:58.278008+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-35.98648012695035, 79.69011850751774, -33.49...",Data_20160509_07_004,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_005,2016_Greenland_P3,"LINESTRING (-33.48977 79.69008, -33.04847 79.6...",{'datetime': '2016-05-09T12:58:56.724961+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-33.48976713364755, 79.58741470499012, -31.07...",Data_20160509_07_005,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_006,2016_Greenland_P3,"LINESTRING (-31.07163 79.58733, -28.72299 79.4...",{'datetime': '2016-05-09T13:04:50.446713+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-31.071625343644197, 79.45484100118856, -28.7...",Data_20160509_07_006,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_007,2016_Greenland_P3,"LINESTRING (-28.72136 79.45474, -26.43312 79.3...",{'datetime': '2016-05-09T13:10:41.311460+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-28.721356594206874, 79.30583599555489, -26.4...",Data_20160509_07_007,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_008,2016_Greenland_P3,"LINESTRING (-26.43153 79.30573, -24.21013 79.1...",{'datetime': '2016-05-09T13:16:40.404466+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-26.431528857175067, 79.13993197131637, -24.2...",Data_20160509_07_008,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature
Data_20160509_07_009,2016_Greenland_P3,"LINESTRING (-24.20859 79.1398, -23.03335 79.04...",{'datetime': '2016-05-09T13:22:41.372318+00:00...,{'CSARP_standard': {'href': 'https://data.cres...,"[-24.20859147990673, 78.9747155545051, -22.024...",Data_20160509_07_009,[],[https://stac-extensions.github.io/file/v2.1.0...,1.1.0,Feature


In [17]:
stac_items.loc[ f'Data_{cresis_fl.iloc[0]["Name"]}' ] 

collection                                         2016_Greenland_P3
geometry           LINESTRING (-22.023075074917752 78.97460752943...
properties         {'datetime': '2016-05-09T13:29:36.934709+00:00...
assets             {'CSARP_standard': {'href': 'https://data.cres...
bbox               [-22.023075074917756, 78.9126981606221, -19.04...
id                                              Data_20160509_07_010
links                                                             []
stac_extensions    [https://stac-extensions.github.io/file/v2.1.0...
stac_version                                                   1.1.0
type                                                         Feature
Name: Data_20160509_07_010, dtype: object